<!--
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0
-->


# Lab 3: Reinforcement Learning from Verifiable Rewards (RLVR)

This lab uses the SFT-trained model from Lab 2 and the evaluator Lambda as a reward function to further improve SQL generation accuracy through execution-based feedback.

In [ ]:
%store -r

In [ ]:
%store

In [ ]:
# Restore session
import boto3
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)
sagemaker_session = Session(sagemaker_client=sm_client)
ROLE = get_execution_role()
DEFAULT_BUCKET = sagemaker_session.default_bucket()
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"

ACCEPT_EULA = True

print(f"Region: {REGION}")
print(f"Bucket: {DEFAULT_BUCKET}")
print(f"Evaluator: {EVALUATOR_ARN}")
print(f"Model Package Group: {MODEL_PACKAGE_GROUP_NAME}")

## Configure SFT + RLVR Trainer

Start from the SFT-trained model (not the base model). The model already knows our schema — RLVR will improve its accuracy through trial and error against the database.

In [ ]:
from sagemaker.train.rlvr_trainer import RLVRTrainer
from sagemaker.core.resources import ModelPackageGroup
from sagemaker.ai_registry.dataset import DataSet

model_package_group = ModelPackageGroup.get(MODEL_PACKAGE_GROUP_NAME)

# Resolve the datasets registered in Lab 1 by name. Pass DataSet objects rather
# than ARNs — the trainer reads .source off the object to sample the data for
# reward function validation, and cannot resolve an ARN to its S3 location.
rlvr_training_dataset = DataSet.get(RLVR_TRAINING_DATASET_NAME)
rlvr_validation_dataset = DataSet.get(RLVR_VALIDATION_DATASET_NAME)
print(f"Training dataset:   {rlvr_training_dataset.source}")
print(f"Validation dataset: {rlvr_validation_dataset.source}")

# Get the latest SFT model package to start RLVR from
response = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1
)
sft_model_package_arn = response['ModelPackageSummaryList'][0]['ModelPackageArn']
print(f"Starting RLVR from SFT model: {sft_model_package_arn}")

sft_rlvr_trainer = RLVRTrainer(
    model=sft_model_package_arn,
    training_dataset=rlvr_training_dataset,
    validation_dataset=rlvr_validation_dataset,
    custom_reward_function=EVALUATOR_ARN,
    s3_output_path=f's3://{DEFAULT_BUCKET}/rlvr-output',
    model_package_group=model_package_group,
    sagemaker_session=sagemaker_session,
    accept_eula=ACCEPT_EULA,
    role=ROLE,
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='rlvr-training',
)

## Set SFT+RLVR Hyperparameters

RLVR needs fundamentally different hyperparameters than SFT — lower learning rate, KL regularization, and multiple rollouts per prompt for the GRPO algorithm to compare.

In [ ]:
from rich.pretty import pprint

# Rollout configuration
sft_rlvr_trainer.hyperparameters.rollout_n = 32
sft_rlvr_trainer.hyperparameters.rollout_temperature = 0.8

# Training loop
sft_rlvr_trainer.hyperparameters.max_epochs = 60
sft_rlvr_trainer.hyperparameters.global_batch_size = 128

# Learning rate — much lower than SFT to prevent policy collapse
sft_rlvr_trainer.hyperparameters.learning_rate = 0.000001
sft_rlvr_trainer.hyperparameters.min_lr = 0.0000001
sft_rlvr_trainer.hyperparameters.lr_warmup_steps_ratio = 0.05

# KL regularization — keeps model anchored to the SFT checkpoint
sft_rlvr_trainer.hyperparameters.use_kl_loss = True
sft_rlvr_trainer.hyperparameters.kl_loss_coef = 0.04

# PPO clipping
sft_rlvr_trainer.hyperparameters.clip_ratio = 0.2
sft_rlvr_trainer.hyperparameters.clip_ratio_low = 0.2
sft_rlvr_trainer.hyperparameters.clip_ratio_high = 0.28

# Prompt and eval settings
sft_rlvr_trainer.hyperparameters.max_prompt_length = 1024
sft_rlvr_trainer.hyperparameters.temperature = 0.0

print("SFT+RLVR hyperparameters:")
pprint(sft_rlvr_trainer.hyperparameters.to_dict())

## Configure Base + RLVR Trainer (Ablation)

To test whether SFT-before-RLVR actually matters, we train a second arm: the same RLVR configuration applied directly to the **base** Llama 3.2 3B, skipping Lab 2's fine-tuning entirely.

Both arms use the same data, the same reward function, the same learning rate, the same KL coefficient, the same 60 steps — **with one deliberate exception**, called out below.

### The one hyperparameter that differs, and why

| | SFT + RLVR | Base + RLVR (ablation) |
|---|---|---|
| Starting model | SFT checkpoint from Lab 2 | Base Llama 3.2 3B Instruct |
| **`rollout_n`** | **32** | **8** |
| Everything else | identical | identical |

The ablation arm generates 8 SQL candidates per prompt instead of 32. This is **a deliberate concession to the workshop's time budget**, not an oversight: generation dominates RLVR wall-clock, and running the second arm at 32 rollouts would roughly quadruple its generation work and push the lab well past its slot. The two arms run in parallel, so the fast arm is what keeps the whole lab inside ~60 minutes.

### What that costs you, stated honestly

`rollout_n` is the **GRPO group size** — the number of samples the algorithm computes *relative* advantage across. That has a real consequence: with 8 rollouts instead of 32, the advantage estimate for each step is computed from a smaller sample and is therefore noisier.

So when you look at the two curves in MLflow, be precise about what you can conclude:

**What this ablation does establish:** the base arm starts substantially lower and stays lower throughout. That gap is large, consistent, and appears from the very first step — before group size has had a chance to matter much. It is the effect of missing SFT, and the conclusion holds.

**What it does not establish:** any fine-grained comparison of curve *smoothness*, entropy behavior, or exact final reward between the arms. Two variables moved, so some of the ablation arm's rougher curves come from the smaller group rather than from the missing SFT. Do not read those differences as purely an SFT effect.

**What a publication-grade version would do:** match `rollout_n` at 32 on both arms, isolating the single variable, at roughly 4× the generation cost on this arm.

That caveat is worth more to you than a clean unexplained result would be. When you read someone else's ablation, "did this experiment actually isolate the variable it claims to?" is the first question to ask — and here you can see exactly why the answer is "mostly, with a known confound."

In [ ]:
# Create a separate model package group for the base+RLVR ablation
BASE_MODEL = "meta-textgeneration-llama-3-2-3b-instruct"
ABLATION_GROUP_NAME = "text-to-sql-3b-rlvr-only"

ablation_model_group = ModelPackageGroup.create(
    model_package_group_name=ABLATION_GROUP_NAME,
    model_package_group_description='RLVR without SFT - ablation study'
)

base_rlvr_trainer = RLVRTrainer(
    model=BASE_MODEL,  # Base model, NOT the SFT checkpoint
    training_dataset=rlvr_training_dataset,  # Same DataSet objects as above
    validation_dataset=rlvr_validation_dataset,
    custom_reward_function=EVALUATOR_ARN,
    s3_output_path=f's3://{DEFAULT_BUCKET}/rlvr-base-output',
    model_package_group=ablation_model_group,
    sagemaker_session=sagemaker_session,
    accept_eula=ACCEPT_EULA,
    role=ROLE,
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='rlvr-base-training',
)

# Identical hyperparameters to the SFT arm, with ONE deliberate exception:
# rollout_n is 8 here instead of 32.
#
# Why: generation dominates RLVR wall-clock, so 32 rollouts on this arm would
# roughly 4x its generation work and push the lab past its time slot. 8 keeps
# the ablation affordable in a workshop.
#
# What it costs: rollout_n is the GRPO group size, so a smaller group means
# noisier advantage estimates. The direction of the result (base arm starts and
# stays lower) is far too large to be explained by group size — but do not
# compare curve smoothness or exact final reward between the arms, because two
# variables moved. See the markdown above.
base_rlvr_trainer.hyperparameters.rollout_n = 8
base_rlvr_trainer.hyperparameters.rollout_temperature = 0.8
base_rlvr_trainer.hyperparameters.max_epochs = 60
base_rlvr_trainer.hyperparameters.global_batch_size = 128
base_rlvr_trainer.hyperparameters.learning_rate = 0.000001
base_rlvr_trainer.hyperparameters.min_lr = 0.0000001
base_rlvr_trainer.hyperparameters.lr_warmup_steps_ratio = 0.05
base_rlvr_trainer.hyperparameters.use_kl_loss = True
base_rlvr_trainer.hyperparameters.kl_loss_coef = 0.04
base_rlvr_trainer.hyperparameters.clip_ratio = 0.2
base_rlvr_trainer.hyperparameters.clip_ratio_low = 0.2
base_rlvr_trainer.hyperparameters.clip_ratio_high = 0.28
base_rlvr_trainer.hyperparameters.max_prompt_length = 1024
base_rlvr_trainer.hyperparameters.temperature = 0.0

print(f"Base+RLVR trainer configured (ablation group: {ABLATION_GROUP_NAME})")

## Why *this* ablation, and not a different one

An ablation costs an hour of GPU time and a second concurrent training job, so the workshop runs exactly one. It is worth being explicit about why this is the one.

**SFT-vs-no-SFT is the only ablation whose result changes what you would do.** The workshop's central claim is that RLVR needs a policy that already emits parseable SQL — otherwise nearly every rollout scores zero, every candidate in the group looks equally bad, GRPO has no relative signal to work with, and training stalls. That claim is the reason Lab 2 exists at all. If it were false, you would skip SFT entirely and save half the workshop. Everything else you could ablate here would tune a result you were going to get anyway; this one decides whether a whole stage belongs in the pipeline.

### What else you could ablate

If you extend this workshop on your own, these are the arms worth running, roughly in order of signal per GPU-hour:

| Ablation | Change | What you should expect |
|---|---|---|
| **KL anchor** | `kl_loss_coef = 0.04` → `0` | Faster early reward gain, then degeneration — the policy drifts from the SFT checkpoint, output formatting decays, and reward eventually collapses. The cheapest way to *see* what the KL term is buying. |
| **Reward weighting** | The `0.3` / `0.7` blend of `execution_success` and `result_set_f1` | Arguably the highest-signal knob in the whole workshop. Try F1-only (no 0.30 floor for merely-parseable SQL) and binary exact-match. Exact-match-only should train worst: every wrong answer scores identically, so there is no gradient between "nearly right" and "nonsense". |
| **`rollout_n` sweep** | 8 → 16 → 32 → 64, holding the starting model fixed | The classic GRPO group-size study, and the one you have already half-seen — the two arms here differ on it. Larger groups give lower-variance advantage estimates at linear cost in generation. This is also the experiment that would *remove* the confound described above. |
| **LoRA rank** | Currently never set anywhere | Lab 2 takes the SDK default and `RLVRTrainer` never specifies a training type at all. Higher rank means more adapter capacity and more to overfit on ~200 examples. |

**Why they are described rather than run:** each additional arm is another ~60 minutes of wall-clock and another concurrent GPU training job. Two arms already saturate the parallelism this lab can use inside its slot.

One thing to notice about the reward-weighting row: that knob lives in the evaluator Lambda you wrote in **Lab 2**, not in this notebook. From inside Lab 3 it is completely invisible — you would have to go back and redeploy the Lambda to change it. That is worth internalizing as a general property of RLVR: **the reward function is a hyperparameter**, usually the most consequential one, and it is often the one furthest from the training code.


## Launch Both Training Jobs

We launch both jobs simultaneously with `wait=False` so they run in parallel. This cuts wall-clock time from ~2 hours (sequential) to ~1 hour.

**If a job fails to start or fails during training.** Both arms use GPU training instances, which are the most common source of trouble on a fresh account:

- **Quota / `ResourceLimitExceeded`** — the account may have no quota for the GPU instance type, or not enough for *two* jobs at once. Check **Service Quotas → Amazon SageMaker** for the relevant `ml.*` *training job usage* quota and request an increase if it is 0 or 1. In a Workshop Studio event account this is pre-provisioned, so it mainly bites when adapting the workshop to your own account. If quota only allows one job at a time, launch the two cells sequentially (start the second once the first reaches `InProgress`) instead of together.
- **`InsufficientCapacityError`** — capacity for that instance type is momentarily unavailable in the Region. This is transient: wait a few minutes and re-run the launch cell.
- **A job that starts then fails** — the monitor cell below will show `Failed`. Open the job under **SageMaker AI → Training → Training jobs**, read its **failure reason**, and check its **CloudWatch Logs** for the stack trace. The most common non-quota cause is a malformed dataset from Lab 1 — which is why Lab 1 validates the RLVR files before registering them. Fix the cause and re-run the launch cell; job names are generated fresh each run, so a retry never collides with the failed job.

In [ ]:
# Launch both training jobs in parallel (non-blocking)
# train() returns the TrainingJob — capture it, the trainer only stores it
# on the private _latest_training_job attribute.
print("Launching SFT+RLVR training...")
sft_training_job = sft_rlvr_trainer.train(wait=False)

print("Launching Base+RLVR (ablation) training...")
base_training_job = base_rlvr_trainer.train(wait=False)

print(f"\nSFT+RLVR job:  {sft_training_job.training_job_name}")
print(f"Base+RLVR job: {base_training_job.training_job_name}")
print("\nBoth jobs submitted. Run the next cell to monitor progress.")

## Monitor Training Jobs

This cell polls both jobs every 60 seconds and displays their status. It blocks until both jobs reach a terminal state (Completed, Failed, or Stopped).

In [ ]:
import time
from IPython.display import clear_output

def get_job_status(job_name):
    """Get training job status from SageMaker."""
    try:
        desc = sm_client.describe_training_job(TrainingJobName=job_name)
        status = desc['TrainingJobStatus']
        secondary = desc.get('SecondaryStatus', '')
        return f"{status} ({secondary})" if secondary else status
    except Exception as e:
        return f"Error: {e}"

# Get job names from the TrainingJob objects returned by train()
sft_job_name = sft_training_job.training_job_name
base_job_name = base_training_job.training_job_name

print(f"SFT+RLVR job:  {sft_job_name}")
print(f"Base+RLVR job: {base_job_name}")
print("\nMonitoring...")

terminal_states = {'Completed', 'Failed', 'Stopped'}

while True:
    sft_desc = sm_client.describe_training_job(TrainingJobName=sft_job_name)
    base_desc = sm_client.describe_training_job(TrainingJobName=base_job_name)
    
    sft_status = sft_desc['TrainingJobStatus']
    base_status = base_desc['TrainingJobStatus']
    sft_secondary = sft_desc.get('SecondaryStatus', '')
    base_secondary = base_desc.get('SecondaryStatus', '')
    
    clear_output(wait=True)
    print("=" * 60)
    print("  RLVR TRAINING MONITOR")
    print("=" * 60)
    print(f"  SFT + RLVR:   {sft_status:12s} {sft_secondary}")
    print(f"  Base + RLVR:  {base_status:12s} {base_secondary}")
    print(f"\n  Last checked: {time.strftime('%H:%M:%S')}")
    
    if sft_status in terminal_states and base_status in terminal_states:
        print("\n" + "=" * 60)
        if sft_status == 'Completed' and base_status == 'Completed':
            print("  BOTH JOBS COMPLETED SUCCESSFULLY")
        else:
            print("  WARNING: One or more jobs did not complete successfully")
            if sft_status != 'Completed':
                print(f"    SFT+RLVR: {sft_status}")
            if base_status != 'Completed':
                print(f"    Base+RLVR: {base_status}")
        print("=" * 60)
        break
    
    time.sleep(60)

## Expected Results

Once both jobs finish, open the `rlvr-training` and `rlvr-base-training` experiments in MLflow and compare the training curves over the 60 steps. The table below shows the range you should expect to see — treat these as reference values, not targets to match exactly.

| Metric | SFT + RLVR | Base + RLVR (Ablation) | Confounded by `rollout_n`? |
|--------|-----------|------------------------|---|
| **Starting reward** | ~0.45 | ~0.39 | No — measured before training |
| **Final reward** | ~0.55 | ~0.49 | Partly |
| **Reward gain** | ~+0.10 | ~+0.10 | Partly |
| **Episode length** | Stable, short (concise SQL) | Noisy and longer (verbose) | Length no, noisiness partly |
| **Policy entropy** | Smooth decline | Noisy, erratic decline | Yes |
| **Mean advantage** | Converges to 0 quickly | Still oscillating near 0 at the end | Yes |

That last column matters. The ablation arm runs at `rollout_n = 8` rather than 32 for the time-budget reason explained earlier, and because `rollout_n` is the GRPO group size, anything to do with the *variance* of the curves is influenced by that as well as by the missing SFT. The rows marked "Yes" are the ones you cannot cleanly attribute to SFT.

### Why your numbers will differ

RLVR is stochastic in several places at once, so no two runs produce identical curves:

- **Rollout sampling.** Each prompt is answered `rollout_n` times at `rollout_temperature = 0.8`. Different samples produce different rewards, which produce different gradients.
- **GRPO advantages are relative.** Advantage is computed *within* each group of rollouts, so a lucky or unlucky batch shifts the update direction for that step.
- **Execution-based rewards are all-or-nothing.** The evaluator runs the generated SQL against the database — a query is either right or wrong, so reward moves in coarse jumps rather than smoothly.
- **Your dataset is your own.** The queries come from `pg_stat_statements` on *your* Aurora cluster and the descriptions were generated by Bedrock, so the training data itself differs from run to run.
- **60 steps is a short run.** With this few gradient steps the run has not fully converged, so where the curve happens to land is somewhat arbitrary.

Expect final rewards within roughly ±0.05 of the values above, and expect the curves to be bumpy rather than monotonic. A step where reward drops is normal.

### What to look for instead

The point of this ablation is the *shape* of the comparison, not the exact numbers. These conclusions should hold in your run:

1. **SFT + RLVR starts higher and finishes higher.** This is the finding, and it is the one the experiment genuinely supports. The starting reward is measured before any gradient step, so group size cannot explain it: SFT gives the model a head start — it already generates schema-valid SQL, so more rollouts earn reward from step 1 and there is more signal to learn from.

2. **The base model still improves noticeably.** Llama 3.2 3B **Instruct** has general SQL knowledge from its own instruction tuning, so it produces some valid queries even without our SFT. That is enough for RLVR to get traction — which is worth noticing, because it means "no SFT" here does not mean "no instruction tuning at all". A raw, non-instruct base model would be starting from much further back, and would plausibly stay near zero reward for lack of any parseable output to earn signal from. **This workshop does not test that** — it is a reasonable expectation from how RLVR works, not a result you are seeing on screen.

3. **The SFT run is visibly more stable — but read this row carefully.** You should see a smoother entropy decline, steadier episode length, and advantage settling toward 0 sooner on the SFT arm. Two things contribute to that, and the experiment does not separate them: SFT gives the policy a consistent output distribution to start from, *and* the SFT arm computes each advantage across 32 rollouts instead of 8, which is inherently a lower-variance estimate. Expect the direction, do not attribute the magnitude entirely to SFT.

4. **Episode length is the clearest tell — and the cleanest one.** The SFT model emits short, bare SQL because SFT taught it the output format. The base model wraps SQL in explanations or markdown fences that the reward function has to strip. Group size does not change how verbose a model is, so the *level* difference here is an SFT effect even though the step-to-step jitter is not. RLVR alone does not reliably teach formatting.

**Bottom line:** SFT provides three things RLVR struggles to learn on its own — a higher starting point for the reward signal, formatting discipline, and training stability. If your run shows the base model matching or beating SFT+RLVR, check the training curves before concluding anything: on a 60-step run that is more likely to be sampling noise than a real reversal.


## Review the training curves in MLflow

Both jobs are done, so the two training experiments are now fully populated. This is the moment to actually look at them — the reward curves are the evidence for everything the section above claims, and the evaluation numbers later in this notebook will not show you *how* the models got there.

The cell below opens MLflow. Compare `rlvr-training` against `rlvr-base-training`, overlaying the reward curves if the UI lets you.


In [ ]:
# Open MLflow to compare the two training runs.
#
# CreatePresignedMlflowAppUrl mints a short-lived signed URL that logs you
# straight in. If it is unavailable in this environment's boto3, fall back to
# the console landing page.
from IPython.display import display, HTML


def mlflow_link(experiments, note=""):
    try:
        url = sm_client.create_presigned_mlflow_app_url(
            Arn=MLFLOW_ARN
        )["AuthorizedUrl"]
        label = "Open MLflow (signed link, valid ~5 minutes)"
    except Exception as e:
        print(f"Presigned URL unavailable ({type(e).__name__}), falling back to console link.")
        url = f"https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/mlflow"
        label = "Open MLflow in the AWS Console"
    display(HTML(f'<a href="{url}" target="_blank"><b>{label}</b></a>'))
    if note:
        print(f"\n{note}")
    for e in experiments:
        print(f"  - {e}")


mlflow_link(
    ["rlvr-training          (SFT + RLVR)", "rlvr-base-training     (Base + RLVR, ablation)"],
    "Training experiments to compare:",
)
print("\nMetrics worth plotting: reward, policy entropy, episode length, mean advantage.")


## Evaluate Both Models

Now that both training jobs are complete, evaluate each against the validation and combined datasets — four evaluations in total.

**Only two evaluation jobs can run at once**, so we run them in two batches of two rather than launching all four together:

1. Launch the SFT+RLVR evaluations (validation + combined), then **block until both finish**.
2. Launch the Base+RLVR evaluations (validation + combined), then block until both finish.

Do not skip the waiting cell between the two batches. Launching all four at once puts the third and fourth jobs over the concurrency limit.


In [ ]:
from sagemaker.train.evaluate import CustomScorerEvaluator

# Get latest model packages from both groups
response = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1
)
rlvr_model_package_arn = response['ModelPackageSummaryList'][0]['ModelPackageArn']
print(f"SFT+RLVR model: {rlvr_model_package_arn}")

response = sm_client.list_model_packages(
    ModelPackageGroupName=ABLATION_GROUP_NAME,
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1
)
ablation_model_package_arn = response['ModelPackageSummaryList'][0]['ModelPackageArn']
print(f"Base+RLVR model: {ablation_model_package_arn}")

In [ ]:
import time
from IPython.display import clear_output

TERMINAL_STATES = {'Succeeded', 'Failed', 'Stopped', 'Completed'}
SUCCESS_STATES = {'Succeeded', 'Completed'}


def wait_for_evaluations(executions, batch_label):
    """Block until every evaluation reaches a terminal state.

    executions maps a display name to the EvaluationPipelineExecution returned
    by CustomScorerEvaluator.evaluate() — NOT the evaluator object itself, which
    has no status to report. Each poll calls refresh() to pull the latest
    pipeline status, then reads status.overall_status. Returns True if all
    succeeded. Only two evaluation jobs may run concurrently, so this is what
    keeps the second batch from starting while the first is still active.
    """
    print(f"Monitoring {batch_label}...\n")

    while True:
        statuses = {}
        for name, execution in executions.items():
            # Let a genuinely broken poll surface as the status text rather than
            # collapsing to a non-terminal sentinel that loops forever.
            try:
                execution.refresh()
                statuses[name] = execution.status.overall_status
            except Exception as e:
                statuses[name] = f"Error: {type(e).__name__}: {e}"

        clear_output(wait=True)
        print("=" * 60)
        print(f"  EVALUATION MONITOR — {batch_label}")
        print("=" * 60)
        for name, status in statuses.items():
            print(f"  {name:30s} {status}")
        print(f"\n  Last checked: {time.strftime('%H:%M:%S')}")

        if all(s in TERMINAL_STATES for s in statuses.values()):
            failed = [n for n, s in statuses.items() if s not in SUCCESS_STATES]
            print("\n" + "=" * 60)
            if not failed:
                print(f"  {batch_label.upper()} COMPLETED SUCCESSFULLY")
            else:
                print(f"  WARNING: Some evaluations in {batch_label} did not succeed:")
                for n in failed:
                    print(f"    {n}: {statuses[n]}")
            print("=" * 60)
            return not failed

        time.sleep(60)

### Batch 1: SFT+RLVR Evaluation

Launch both SFT+RLVR evaluations. That is two concurrent jobs — the limit — so the Base+RLVR pair has to wait for these to finish.


In [ ]:
# RLVR_VALIDATION_DATASET_ARN and RLVR_COMBINED_DATASET_ARN (used across the
# four evaluation cells below) are stored by Lab 1 alongside the *_NAME
# variables, and restored by the %store -r at the top of this notebook. If a
# NameError fires on either, run 01-data-preparation.ipynb to completion first.
rlvr_evaluation = CustomScorerEvaluator(
    evaluator=EVALUATOR_ARN,
    dataset=RLVR_VALIDATION_DATASET_ARN,
    model=rlvr_model_package_arn,
    s3_output_path=f's3://{DEFAULT_BUCKET}/rlvr-evaluation',
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='rlvr-eval-validation',
    model_package_group=model_package_group,
    # Score only this fine-tuned model. Lab 2 already scored the base Llama 3.2 3B
    # on both datasets, and Lab 4 pulls that row into the final comparison — so a
    # base run here would only duplicate an existing row and spend another
    # evaluation slot doing it. All four evaluations in this lab set this
    # explicitly so the four cells read identically.
    evaluate_base_model=False,
)

# Capture the returned execution — that is the object with a live status.
# The evaluator itself has no status to poll.
sft_val_execution = rlvr_evaluation.evaluate()
print("Launched SFT+RLVR validation evaluation -> rlvr-eval-validation.")


In [ ]:
combined_rlvr_evaluation = CustomScorerEvaluator(
    evaluator=EVALUATOR_ARN,
    dataset=RLVR_COMBINED_DATASET_ARN,
    model=rlvr_model_package_arn,
    s3_output_path=f's3://{DEFAULT_BUCKET}/rlvr-evaluation-combined',
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='rlvr-eval-combined',
    model_package_group=model_package_group,
    # Score only this fine-tuned model. Lab 2 already scored the base Llama 3.2 3B
    # on both datasets, and Lab 4 pulls that row into the final comparison — so a
    # base run here would only duplicate an existing row and spend another
    # evaluation slot doing it. All four evaluations in this lab set this
    # explicitly so the four cells read identically.
    evaluate_base_model=False,
)

sft_combined_execution = combined_rlvr_evaluation.evaluate()

print("Both SFT+RLVR evaluations launched. Run the next cell to wait for them")
print("before launching the Base+RLVR pair.")


### Wait for Batch 1

This cell blocks until both SFT+RLVR evaluations reach a terminal state, freeing up the two concurrency slots. **Do not skip it** — running the Base+RLVR cells while these are still active exceeds the limit and those jobs will fail to start.


In [ ]:
sft_batch_ok = wait_for_evaluations(
    {
        "SFT+RLVR (validation)": sft_val_execution,
        "SFT+RLVR (combined)": sft_combined_execution,
    },
    "Batch 1: SFT+RLVR",
)

# Both slots are now free. Proceed to the Base+RLVR batch below even if one of
# these failed — the ablation is independent, and a partial comparison is more
# useful than none. Note any failure so you can interpret Lab 4's table.
if not sft_batch_ok:
    print("\nNote: a Batch 1 evaluation did not succeed. Its row will be missing")
    print("from the Lab 4 comparison table.")

### Batch 2: Base+RLVR (Ablation) Evaluation

Batch 1 has finished, so both concurrency slots are free. Launch the two ablation evaluations.


In [ ]:
base_rlvr_evaluation = CustomScorerEvaluator(
    evaluator=EVALUATOR_ARN,
    dataset=RLVR_VALIDATION_DATASET_ARN,
    model=ablation_model_package_arn,
    s3_output_path=f's3://{DEFAULT_BUCKET}/rlvr-base-evaluation',
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='rlvr-ablation-eval-validation',
    model_package_group=ablation_model_group,
    # Score only this fine-tuned model. Lab 2 already scored the base Llama 3.2 3B
    # on both datasets, and Lab 4 pulls that row into the final comparison — so a
    # base run here would only duplicate an existing row and spend another
    # evaluation slot doing it. All four evaluations in this lab set this
    # explicitly so the four cells read identically.
    evaluate_base_model=False,
)

base_val_execution = base_rlvr_evaluation.evaluate()
print("Launched Base+RLVR validation evaluation -> rlvr-ablation-eval-validation.")


In [ ]:
base_rlvr_combined_eval = CustomScorerEvaluator(
    evaluator=EVALUATOR_ARN,
    dataset=RLVR_COMBINED_DATASET_ARN,
    model=ablation_model_package_arn,
    s3_output_path=f's3://{DEFAULT_BUCKET}/rlvr-base-evaluation-combined',
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='rlvr-ablation-eval-combined',
    model_package_group=ablation_model_group,
    # Score only this fine-tuned model. Lab 2 already scored the base Llama 3.2 3B
    # on both datasets, and Lab 4 pulls that row into the final comparison — so a
    # base run here would only duplicate an existing row and spend another
    # evaluation slot doing it. All four evaluations in this lab set this
    # explicitly so the four cells read identically.
    evaluate_base_model=False,
)

base_combined_execution = base_rlvr_combined_eval.evaluate()

print("Both Base+RLVR evaluations launched. Run the next cell to wait for them.")


### Wait for Batch 2

Block until both ablation evaluations finish. Once this cell returns, all four evaluations are complete and Lab 4 has everything it needs.


In [ ]:
base_batch_ok = wait_for_evaluations(
    {
        "Base+RLVR (validation)": base_val_execution,
        "Base+RLVR (combined)": base_combined_execution,
    },
    "Batch 2: Base+RLVR",
)

print("\n" + "=" * 60)
if sft_batch_ok and base_batch_ok:
    print("  ALL FOUR EVALUATIONS COMPLETED SUCCESSFULLY")
    print("  Ready for Lab 4.")
else:
    print("  WARNING: Not all evaluations succeeded.")
    print(f"    Batch 1 (SFT+RLVR):  {'OK' if sft_batch_ok else 'see above'}")
    print(f"    Batch 2 (Base+RLVR): {'OK' if base_batch_ok else 'see above'}")
    print("  Lab 4 will still run, but missing runs will be absent from its tables.")
print("=" * 60)

## All four evaluations are in MLflow

Every number Lab 4 needs from this lab now exists. Open MLflow once more to see the evaluation scores before moving on — `aggregate_reward` is the ranking metric, and this is the first place you can see whether RLVR actually improved on the SFT checkpoint or merely matched it.


In [ ]:
mlflow_link(
    [
        "rlvr-eval-validation             (SFT + RLVR, held-out set)",
        "rlvr-eval-combined               (SFT + RLVR, train + validation)",
        "rlvr-ablation-eval-validation    (Base + RLVR, held-out set)",
        "rlvr-ablation-eval-combined      (Base + RLVR, train + validation)",
    ],
    "Evaluation experiments written by this lab:",
)
print("\nRank by aggregate_reward. Lab 4 pulls all four of these into one table,")
print("alongside the base and SFT rows from Lab 2 and the frontier models.")


In [ ]:
# Persist for Lab 4 comparison
ABLATION_GROUP_NAME = "text-to-sql-3b-rlvr-only"
%store ABLATION_GROUP_NAME